# Man-in-the-Middle — Hijack-Pursuit

A **target** drone (green) flies its own north mission. A **victim** drone
(red) starts a short mission of its own, but an attacker has a
man-in-the-middle interposed on the victim's GCS↔logic link.
Mid-mission the MITM switches the victim to GUIDED and streams position
setpoints that continuously chase the target's live Remote ID position —
the same behaviour as `PursuitPlan`, except the guided commands originate
from the attacker, not from a cooperative plan.

The attacker learns the target's position **passively**: the MITM overhears
the very Remote ID feed the compromised drone already receives, so the
Oracle needs no changes. (The target must stay within the Oracle's
transmission range of the victim for its Remote ID to reach the feed — the
same constraint `PursuitPlan` lives under.)

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan
from simulator.runtime.mitm import HijackPursuitStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin, layout, and missions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

speed = 5.0        # m/s
cruise_alt = 10.0  # m — both drones share one altitude layer
model = Model.IRIS

target_sysid = 1   # green — flies a normal mission, gets chased
victim_sysid = 2   # red — hijacked by the MITM to pursue the target

# Homes: side by side on the same line (both at north=0), ~20 m apart in
# east. Flying abreast means the hijacked drone starts level with its target
# and can catch up, rather than trailing from behind. The lateral gap keeps
# the target's Remote ID well within transmission range of the victim.
target_home = ENUPose(-10.0, 0.0, 0.0, 0)
victim_home = ENUPose(10.0, 0.0, 0.0, 0)

# Both drones fly the same-length north mission, side by side; then the MITM
# hijacks the victim at seq=2 (right after takeoff/first waypoint) into
# pursuit of the target.
target_wps = [ENU(0, 0, 0), ENU(0, 40, cruise_alt), ENU(0, 80, cruise_alt)]
victim_wps = [ENU(0, 0, 0), ENU(0, 40, cruise_alt), ENU(0, 80, cruise_alt)]
trigger_seq = 2

## Vehicles

In [ ]:
mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

target_plan = AutoPlan.from_relative_path(
    name="target_mission",
    sysid=target_sysid,
    gra_origin=gra_origin,
    relative_home=target_home,
    relative_path=target_wps,
    mission_path=str(mission_folder / "mitm_pursuit_target.waypoints"),
    navigation_speed=speed,
    firmware=model.firmware,
)
target = SimVehicle.from_relative(
    model=model,
    sysid=target_sysid,
    color=Color.GREEN,
    plan=target_plan,
    enu_origin=enu_origin,
    relative_home=target_home,
    relative_path=target_wps,
)

victim_plan = AutoPlan.from_relative_path(
    name="victim_mission",
    sysid=victim_sysid,
    gra_origin=gra_origin,
    relative_home=victim_home,
    relative_path=victim_wps,
    mission_path=str(mission_folder / "mitm_pursuit_victim.waypoints"),
    navigation_speed=speed,
    firmware=model.firmware,
)

# The MITM is interposed on ONLY the victim. On trigger it switches the victim
# to GUIDED and streams setpoints chasing the target's overheard Remote ID.
pursuit = HijackPursuitStrategy(
    target_sysid=target_sysid,
    trigger_seq=trigger_seq,
    update_interval=0.5,
)

# `avoidance=False` disables the victim's ADS-B avoidance so it actually closes
# on the target instead of dodging its beacon. The target keeps avoidance on
# (the default) — so with avoidance it will try to flee as the victim nears,
# turning this into a pursuit-evasion. Flip the target's flag to compare.
victim = SimVehicle.from_relative(
    model=model,
    sysid=victim_sysid,
    color=Color.RED,
    plan=victim_plan,
    enu_origin=enu_origin,
    relative_home=victim_home,
    relative_path=victim_wps,
    mitm=pursuit,
    avoidance=False,
)

# One shared GCS process monitors both drones.
gcs = SimGCS(name=f"PURSUIT_{Color.BLUE.emoji}")
gcs.own(target)
gcs.own(victim)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Oracle

In [ ]:
orac = Oracle()
orac.add_vehicle(target)
orac.add_vehicle(victim)

## Simulator + MITM hijack-pursuit

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=1,
    terminals=[SimProcess.MITM, SimProcess.LOGIC, SimProcess.GCS],
    speedup=1,
)
simulator.preview()

In [ ]:
simulator.run(timeout=120)

## What to observe

- **Green (target)** flies its own north mission, unaware of the attack.
- **Red (victim)** begins a short north mission, then — with no command from
  its real GCS — breaks off and chases the green drone, continuously
  re-targeting the green drone's live Remote ID position.
- `simulator/logs/mitm/mitm_2.log` — `MITM hijack-pursuit: vehicle 2 -> pursuing
  sysid 1` and the stream of `redirecting vehicle 2 to target 1 at (...)` lines.
- The pursuit only tracks while the target stays within transmission range
  of the victim; if the target flies out of range the setpoints freeze at
  its last-heard position until it is heard again.